# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [21]:
#loading the dataset
df = pd.read_csv('data/AviationData.csv', low_memory=False, encoding='latin-1')
print(df.shape)
df.head()


(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [22]:
#checking for null values
df.isnull().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [23]:
#checking the data types of the columns
df.dtypes

Event.Id                   object
Investigation.Type         object
Accident.Number            object
Event.Date                 object
Location                   object
Country                    object
Latitude                   object
Longitude                  object
Airport.Code               object
Airport.Name               object
Injury.Severity            object
Aircraft.damage            object
Aircraft.Category          object
Registration.Number        object
Make                       object
Model                      object
Amateur.Built              object
Number.of.Engines         float64
Engine.Type                object
FAR.Description            object
Schedule                   object
Purpose.of.flight          object
Air.carrier                object
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition          object
Broad.phase.of.flight      object
Report.Status 

In [24]:
#summary statistics of the dataset
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

- inspecting relevant columns (looking for accidents and aircraft category airplane)

In [25]:
print("Investigation Type:")
print(df['Investigation.Type'].value_counts())

Investigation Type:
Investigation.Type
Accident    85015
Incident     3874
Name: count, dtype: int64


In [26]:
print("Aircraft Category:")
print(df['Aircraft.Category'].value_counts(dropna=False))

Aircraft Category:
Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64


In [27]:
print("Amateur Built:")
print(df['Amateur.Built'].value_counts(dropna=False))

Amateur Built:
Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64


In [ ]:
#inspecting the Event Date column
print("Event Date")
print(df['Event.Date'].head(10))
print()
print("Date range:")
print(df['Event.Date'].min())
print(df['Event.Date'].max())

Event Date
0    1948-10-24
1    1962-07-19
2    1974-08-30
3    1977-06-19
4    1979-08-02
5    1979-09-17
6    1981-08-01
7    1982-01-01
8    1982-01-01
9    1982-01-01
Name: Event.Date, dtype: object

Date range:
1948-10-24
2022-12-29


In [29]:
# Convert 'Event.Date' to datetime, coercing errors to NaT
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter 1: Keep only 'Accident' investigation types
df = df[df['Investigation.Type'] == 'Accident'].copy()
print(f"Accidents only: {len(df)}")

# Filter 2: Keep only airplanes (exclude helicopters, gliders, balloons, etc.)
non_airplane = ['Helicopter', 'Glider', 'Balloon', 'Gyrocraft', 'Weight-Shift',
                'Powered Parachute', 'Ultralight', 'Blimp', 'Powered-Lift', 
                'Rocket', 'ULTR', 'WSFT']
df = df[~df['Aircraft.Category'].isin(non_airplane)].copy()
print(f"Airplanes only: {len(df)}")

# Filter 3: Remove amateur-built aircraft
df = df[df['Amateur.Built'] != 'Yes'].copy()
print(f"Professional builds only: {len(df)}")

# Filter 4: From 1983 onwards (to focus on more recent data and reduce missing values)
df = df[df['Event.Date'].dt.year >= 1983].copy()
print(f"From 1983 onwards: {len(df)}")

Accidents only: 85015
Airplanes only: 80457
Professional builds only: 72381
From 1983 onwards: 69488


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

-here we'll check for missing values in the injury columns before we impute relevant columns

In [30]:
# Inspect injury columns
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 
               'Total.Minor.Injuries', 'Total.Uninjured']

print("Missing values:")
print(df[injury_cols].isnull().sum())

print("Summary:")
df[injury_cols].describe()

Missing values:
Total.Fatal.Injuries       9076
Total.Serious.Injuries    10067
Total.Minor.Injuries       9603
Total.Uninjured            4645
dtype: int64
Summary:


,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,60412.000000,59421.000000,59885.000000,64843.000000
mean,0.722721,0.289258,0.370894,3.692473
std,6.151081,1.667902,1.905800,21.637605
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,1.000000
75%,0.000000,0.000000,0.000000,2.000000
max,349.000000,161.000000,200.000000,699.000000


- The NaN values probably mean 0 passengers were fatally injured, so I'll fill them with 0

In [31]:
# Fill NaN injury values with 0
# Assumption: missing values likely mean no injuries in that category
for col in injury_cols:
    df[col] = df[col].fillna(0)

# Estimating total people on board
df['Total.OnBoard'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] +
                       df['Total.Minor.Injuries'] + df['Total.Uninjured'])

# Removing rows where total on board is 0 
df = df[df['Total.OnBoard'] > 0].copy()

# Create injury rate: fraction fatally or seriously injured
df['Injury.Rate'] = ((df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / 
                      df['Total.OnBoard'])

print(f"Remaining rows: {len(df)}")
print("Injury Rates:")
df['Injury.Rate'].describe()

Remaining rows: 69103
Injury Rates:


count    69103.000000
mean         0.282111
std          0.432911
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: Injury.Rate, dtype: float64

- From the mean an average of 28% of people on board are injured

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# Inspecting the Aircraft.damage column
print(df['Aircraft.damage'].value_counts(dropna=False))

Aircraft.damage
Substantial    52318
Destroyed      14917
NaN             1222
Minor            582
Unknown           64
Name: count, dtype: int64


In [33]:
# Replacing the  'Unknown' with NaN
df['Aircraft.damage'] = df['Aircraft.damage'].replace('Unknown', np.nan)

# Create binary Is.Destroyed column: 1 = Destroyed, 0 = everything else
df['Is.Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(float)

print(df['Is.Destroyed'].value_counts(dropna=False))

Is.Destroyed
0.0    54186
1.0    14917
Name: count, dtype: int64


- There are 14,917 destroyed aircrafts

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

- Inspecting the make column first

In [34]:
print(f"Unique Makes: {df['Make'].nunique()}")

print("Top 20 Makes:")
print(df['Make'].value_counts().head(20))

Unique Makes: 1816
Top 20 Makes:
Make
Cessna            20592
Piper             11135
CESSNA             4751
Beech              3906
PIPER              2773
Bell               1733
Mooney             1012
BEECH               997
Grumman             974
Boeing              836
Bellanca            816
Hughes              679
Robinson            655
Air Tractor         575
Schweizer           541
Aeronca             456
Maule               425
Champion            412
BOEING              383
Aero Commander      335
Name: count, dtype: int64


- Some makes are the same but counted separately. Next step is to standardize the columns 

In [35]:
# Standardise Make: strip whitespace and convert to title case
df['Make'] = df['Make'].str.strip().str.title()

print(f"Unique Makes: {df['Make'].nunique()}")
print("Top 20 Makes:")
print(df['Make'].value_counts().head(20))

Unique Makes: 1501
Top 20 Makes:
Make
Cessna            25343
Piper             13908
Beech              4903
Bell               1745
Mooney             1247
Boeing             1219
Grumman            1051
Bellanca            972
Hughes              680
Robinson            672
Air Tractor         666
Aeronca             605
Maule               569
Schweizer           547
Champion            502
Stinson             418
Aero Commander      403
Luscombe            390
Taylorcraft         366
North American      364
Name: count, dtype: int64


- Looking for aircrafts with 50 accidents or more , 50 being our minimum threshold for statistical robustness

In [36]:
# Keep only makes with >= 50 accidents
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)].copy()

print(f"Makes with >= 50 accidents: {len(valid_makes)}")
print(f"Rows remaining: {len(df)}")

Makes with >= 50 accidents: 79
Rows remaining: 63760


There are 79 makes with 50 accident and above

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [37]:
print(f"Model NaN count: {df['Model'].isnull().sum()}")
print("Sample models:")
print(df['Model'].value_counts().head(20))

Model NaN count: 14
Sample models:
Model
152          2212
172          1628
172N         1088
PA-28-140     863
172M          753
150           742
172P          659
182           608
180           595
150M          549
PA-18-150     549
PA-18         548
PA-28-180     546
PA-28-161     519
PA-28-181     504
150L          433
A36           424
G-164A        422
PA-38-112     420
140           385
Name: count, dtype: int64


-Creating a derived column that is a unique identifier for a given plane type since the model names are not unique

In [40]:
# Drop the 14 NaN models
df = df.dropna(subset=['Model']).copy()

# Standardising Model
df['Model'] = df['Model'].str.strip().str.upper()

# Creating unique Make.Model identifier
df['Make.Model'] = df['Make'] + ' ' + df['Model']

print(f"Rows remaining: {len(df)}")
print(f"Unique Make.Model : {df['Make.Model'].nunique()}")
print("Sample values:")
print(df['Make.Model'].value_counts().head(10))

Rows remaining: 63746
Unique Make.Model : 5557
Sample values:
Make.Model
Cessna 152         2212
Cessna 172         1628
Cessna 172N        1088
Piper PA-28-140     863
Cessna 172M         753
Cessna 150          742
Cessna 172P         659
Cessna 182          608
Cessna 180          594
Cessna 150M         549
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

- Inspecting engine type

In [41]:
print(df['Engine.Type'].value_counts(dropna=False))

Engine.Type
Reciprocating    53309
NaN               3267
Turbo Prop        2444
Turbo Shaft       2018
Unknown           1306
Turbo Fan         1095
Turbo Jet          306
UNK                  1
Name: count, dtype: int64


- UNK probably stands for unknown, replace with NaN

In [42]:
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan})

print(df['Engine.Type'].value_counts(dropna=False))

Engine.Type
Reciprocating    53309
NaN               4574
Turbo Prop        2444
Turbo Shaft       2018
Turbo Fan         1095
Turbo Jet          306
Name: count, dtype: int64


- Inspecting weather condition

In [43]:
print(df['Weather.Condition'].value_counts(dropna=False))

Weather.Condition
VMC    56263
IMC     4934
NaN     1814
UNK      589
Unk      146
Name: count, dtype: int64


UNk and Unk, are all unknown. Solve this by replacing with Nan

In [ ]:
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan, 'Unk': np.nan})
print(df['Weather.Condition'].value_counts(dropna=False))

Weather.Condition
VMC    56263
IMC     4934
NaN     2549
Name: count, dtype: int64


- Inspecting number of engines

In [45]:
print(df['Number.of.Engines'].value_counts(dropna=False))

Number.of.Engines
1.0    52004
2.0     7840
NaN     3047
0.0      482
4.0      198
3.0      175
Name: count, dtype: int64


In [46]:
# 0 engines for 482 accidents is impossible - replace with NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0.0, np.nan)
print(df['Number.of.Engines'].value_counts(dropna=False))

Number.of.Engines
1.0    52004
2.0     7840
NaN     3529
4.0      198
3.0      175
Name: count, dtype: int64


- Inspecting purpose of flight and broad phase of flight

In [47]:
print(df['Purpose.of.flight'].value_counts(dropna=False))

Purpose.of.flight
Personal                     35976
Instructional                 8757
Unknown                       4187
Aerial Application            3892
Business                      3217
NaN                           2867
Positioning                   1216
Other Work Use                 884
Ferry                          605
Public Aircraft                599
Aerial Observation             576
Executive/corporate            372
Skydiving                      174
Flight Test                    123
Banner Tow                      97
Public Aircraft - Federal       47
Glider Tow                      34
Public Aircraft - State         30
Firefighting                    20
Air Race/show                   17
Public Aircraft - Local         16
Air Race show                   16
External Load                   12
Air Drop                         6
PUBS                             3
ASHO                             3
Name: count, dtype: int64


- Replace Unknown with NaN and standardise Air Race entries

In [49]:
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Unknown': np.nan,
    'Air Race/show': 'Air Race Show',
    'Air Race show': 'Air Race Show'
})

print(df['Purpose.of.flight'].value_counts(dropna=False))

Purpose.of.flight
Personal                     35976
Instructional                 8757
NaN                           7054
Aerial Application            3892
Business                      3217
Positioning                   1216
Other Work Use                 884
Ferry                          605
Public Aircraft                599
Aerial Observation             576
Executive/corporate            372
Skydiving                      174
Flight Test                    123
Banner Tow                      97
Public Aircraft - Federal       47
Glider Tow                      34
Air Race Show                   33
Public Aircraft - State         30
Firefighting                    20
Public Aircraft - Local         16
External Load                   12
Air Drop                         6
PUBS                             3
ASHO                             3
Name: count, dtype: int64


In [50]:
print(df['Broad.phase.of.flight'].value_counts(dropna=False))

Broad.phase.of.flight
NaN            15973
Landing        12539
Takeoff         9409
Cruise          8011
Maneuvering     6094
Approach        4954
Taxi            1493
Climb           1491
Descent         1441
Go-around       1169
Standing         719
Unknown          381
Other             72
Name: count, dtype: int64


In [51]:
# Replace Unknown with NaN
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace('Unknown', np.nan)
print(df['Broad.phase.of.flight'].value_counts(dropna=False))

Broad.phase.of.flight
NaN            16354
Landing        12539
Takeoff         9409
Cruise          8011
Maneuvering     6094
Approach        4954
Taxi            1493
Climb           1491
Descent         1441
Go-around       1169
Standing         719
Other             72
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

- Check for missing values in percentage to decide which columns to drop

In [52]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_pct.round(2))

Schedule                  87.27
Air.carrier               84.78
FAR.Description           73.37
Aircraft.Category         73.33
Longitude                 64.77
Latitude                  64.76
Airport.Code              42.20
Airport.Name              39.40
Broad.phase.of.flight     25.65
Publication.Date          17.67
Purpose.of.flight         11.07
Engine.Type                7.18
Number.of.Engines          5.54
Report.Status              4.82
Weather.Condition          4.00
Aircraft.damage            1.70
Registration.Number        1.26
Country                    0.27
Amateur.Built              0.06
Location                   0.05
Total.Minor.Injuries       0.00
Total.OnBoard              0.00
Injury.Rate                0.00
Is.Destroyed               0.00
Total.Uninjured            0.00
Event.Id                   0.00
Total.Serious.Injuries     0.00
Total.Fatal.Injuries       0.00
Investigation.Type         0.00
Model                      0.00
Make                       0.00
Injury.S

- I'll drop columns with a high percentage of missing data and irrelevant columns

In [53]:
cols_to_drop = ['Schedule', 'Air.carrier', 'FAR.Description', 'Aircraft.Category',
                'Longitude', 'Latitude', 'Airport.Code', 'Airport.Name',
                'Publication.Date', 'Report.Status', 'Registration.Number',
                'Location', 'Country']

df = df.drop(columns=cols_to_drop)
print(f"Remaining columns: {df.columns.tolist()}")
print(f"Shape: {df.shape}")

Remaining columns: ['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Injury.Severity', 'Aircraft.damage', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Total.OnBoard', 'Injury.Rate', 'Is.Destroyed', 'Make.Model']
Shape: (63746, 22)


In [54]:
df['Year'] = df['Event.Date'].dt.year

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [55]:
df.to_csv('data/aviation_cleaned.csv', index=False)
print(f"Shape: {df.shape}")
print("Columns:", df.columns.tolist())

Shape: (63746, 23)
Columns: ['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Injury.Severity', 'Aircraft.damage', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Total.OnBoard', 'Injury.Rate', 'Is.Destroyed', 'Make.Model', 'Year']
